In [1]:
!pip install huggingface_hub
!pip install hf_xet
!pip install -U transformers bitsandbytes
!pip install sentence-transformers
!pip install evaluate
!pip install trl
!pip install -U peft

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 90.5 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 24.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 91.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 68.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# Librerie
import numpy as np
import pandas as pd
import torch
import json
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from tqdm.notebook import tqdm
from datasets import Dataset
from huggingface_hub import notebook_login
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.model_selection import train_test_split

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    AutoModel,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    pipeline
)

from trl import SFTTrainer, SFTConfig
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from sentence_transformers import SentenceTransformer
from textblob import TextBlob

from huggingface_hub.hf_api import HfFolder
HfFolder.save_token("TOKEN")

#notebook_login()
#token= TOKEN
#token= TOKEN

2025-07-14 14:34:40.911879: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752503681.121868      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752503681.186015      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [20]:
 
model_id = "Qwen/Qwen2.5-3B-Instruct"

torch_dtype = torch.bfloat16

 
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=True,
)

# Define model init arguments
model_kwargs = dict(
    attn_implementation='eager', 
    torch_dtype=torch_dtype, 
    device_map="balanced", 
    quantization_config=bnb_config
)

 
model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
tokenizer = AutoTokenizer.from_pretrained(model_id, device_map='balanced') 

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [17]:
torch.cuda.empty_cache()

In [8]:
df = pd.read_csv('/kaggle/input/mental-health/train_with_classification.csv')

unique_questions = df['Context'].unique()

train_questions, test_questions = train_test_split(unique_questions, test_size=0.25, random_state=42)

train_dataset = df[df['Context'].isin(train_questions)]
test_dataset = df[df['Context'].isin(test_questions)]

 
td = Dataset.from_pandas(train_dataset)

td_clean = td.filter(lambda example: example['Response'] is not None and example['Response'] != '')

Filter:   0%|          | 0/2562 [00:00<?, ? examples/s]

In [9]:
def create_conversation(sample):
    context = sample['Context']
    response = sample['Response']

    return {
        "messages": [
            {"role": "user", "content": context},
            {"role": "assistant", "content": response}
        ]
    }

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for convo in convos]
    return { "text" : texts, }

In [10]:
 
train_dataset = td_clean.map(create_conversation, batched=False)
train_dataset = train_dataset.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/2562 [00:00<?, ? examples/s]

Map:   0%|          | 0/2562 [00:00<?, ? examples/s]

In [11]:
from peft import LoraConfig

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=64,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
    modules_to_save=["lm_head", "embed_tokens"] 
)

from trl import SFTConfig

args = SFTConfig(
    dataset_text_field = 'text', 
    output_dir="/kaggle/working/",
    max_seq_length=512,                     
    packing=True,                           
    num_train_epochs=3,                     
    per_device_train_batch_size=1,          
    gradient_accumulation_steps=8,         
    gradient_checkpointing=True,            
    optim="paged_adamw_32bit",
    logging_steps=1,                        
    learning_rate=2e-4,                     
    bf16=True if torch_dtype == torch.bfloat16 else False,   
    max_grad_norm=0.3,                      
    warmup_ratio=0.03,                      
    lr_scheduler_type="linear",
    report_to="none"
)

In [21]:
from trl import SFTTrainer

 
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    peft_config=peft_config,
    processing_class=tokenizer
)

/usr/local/lib/python3.11/dist-packages/trl/trainer/sft_trainer.py:412: UserWarning: Padding-free training is enabled, but the attention implementation is not set to 'flash_attention_2'. Padding-free training flattens batches into a single sequence, and 'flash_attention_2' is the only known attention mechanism that reliably supports this. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation='flash_attention_2'` in the model configuration, or verify that your attention mechanism can handle flattened sequences.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/trl/trainer/sft_trainer.py:458: UserWarning: You are using packing, but the attention implementation is not set to 'flash_attention_2'. Packing flattens batches into a single sequence, and 'flash_attention_2' is the only known attention mechanism that reliably supports this. Using other implementations may lead to cross-contamination between batches. To avoid this, ei

Tokenizing train dataset:   0%|          | 0/2562 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/2562 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [22]:
torch.cuda.empty_cache()

trainer.train()

Step,Training Loss


KeyboardInterrupt: 